In [0]:
-- CHECK 1: All 8 source tables ingested into Bronze
CREATE OR REPLACE TEMP VIEW dq_check_1 AS
WITH bronze_counts AS (
  SELECT 'airlines' AS src_table, COUNT(*) AS row_count FROM sky_dev.bronze.airlines
  UNION ALL
  SELECT 'bookings', COUNT(*) FROM sky_dev.bronze.bookings
  UNION ALL
  SELECT 'flight_bookings', COUNT(*) FROM sky_dev.bronze.flight_bookings
  UNION ALL
  SELECT 'airline_routes', COUNT(*) FROM sky_dev.bronze.airline_routes
  UNION ALL
  SELECT 'passengers', COUNT(*) FROM sky_dev.bronze.passengers
  UNION ALL
  SELECT 'nations', COUNT(*) FROM sky_dev.bronze.nations
  UNION ALL
  SELECT 'regions', COUNT(*) FROM sky_dev.bronze.regions
  UNION ALL
  SELECT 'routes', COUNT(*) FROM sky_dev.bronze.routes
)
SELECT
  'Check 1: Bronze Ingestion' AS check_name,
  CASE WHEN SUM(CASE WHEN row_count = 0 THEN 1 ELSE 0 END) = 0 THEN 'PASS' ELSE 'FAIL' END AS status,
  CONCAT_WS(', ', COLLECT_LIST(CONCAT(src_table, ': ', CAST(row_count AS STRING)))) AS details,
  'CRITICAL' AS severity
FROM bronze_counts;

SELECT * FROM dq_check_1;

In [0]:
-- CHECK 2: Primary keys populated and unique in Silver
CREATE OR REPLACE TEMP VIEW dq_check_2 AS
WITH pk_checks AS (
  SELECT 'airlines.al_airlineid' AS pk_col,
         SUM(CASE WHEN al_airlineid IS NULL THEN 1 ELSE 0 END) AS nulls,
         COUNT(*) - COUNT(DISTINCT al_airlineid) AS dups
  FROM sky_dev.silver.airlines
  UNION ALL
  SELECT 'bookings.bk_bookingid',
         SUM(CASE WHEN bk_bookingid IS NULL THEN 1 ELSE 0 END),
         COUNT(*) - COUNT(DISTINCT bk_bookingid)
  FROM sky_dev.silver.bookings
  UNION ALL
  SELECT 'passengers.pa_passengerid',
         SUM(CASE WHEN pa_passengerid IS NULL THEN 1 ELSE 0 END),
         COUNT(*) - COUNT(DISTINCT pa_passengerid)
  FROM sky_dev.silver.passengers
  UNION ALL
  SELECT 'nations.n_nationkey',
         SUM(CASE WHEN n_nationkey IS NULL THEN 1 ELSE 0 END),
         COUNT(*) - COUNT(DISTINCT n_nationkey)
  FROM sky_dev.silver.nations
  UNION ALL
  SELECT 'regions.r_regionkey',
         SUM(CASE WHEN r_regionkey IS NULL THEN 1 ELSE 0 END),
         COUNT(*) - COUNT(DISTINCT r_regionkey)
  FROM sky_dev.silver.regions
  UNION ALL
  SELECT 'routes.ro_routeid',
         SUM(CASE WHEN ro_routeid IS NULL THEN 1 ELSE 0 END),
         COUNT(*) - COUNT(DISTINCT ro_routeid)
  FROM sky_dev.silver.routes
)
SELECT
  'Check 2: Primary Key Integrity' AS check_name,
  CASE WHEN SUM(nulls) = 0 AND SUM(dups) = 0 THEN 'PASS' ELSE 'FAIL' END AS status,
  CONCAT_WS(', ', COLLECT_LIST(CONCAT(pk_col, ' (nulls:', CAST(nulls AS STRING), ', dups:', CAST(dups AS STRING), ')'))) AS details,
  'CRITICAL' AS severity
FROM pk_checks;

SELECT * FROM dq_check_2;

In [0]:
-- CHECK 3: Fact table contains no null dimension key columns
CREATE OR REPLACE TEMP VIEW dq_check_3 AS
SELECT
  'Check 3: No Null FK Keys in Fact' AS check_name,
  CASE WHEN SUM(CASE WHEN passenger_id IS NULL THEN 1 ELSE 0 END)
        + SUM(CASE WHEN route_id IS NULL THEN 1 ELSE 0 END)
        + SUM(CASE WHEN airline_id IS NULL THEN 1 ELSE 0 END)
        + SUM(CASE WHEN nationkey IS NULL THEN 1 ELSE 0 END) = 0
       THEN 'PASS' ELSE 'FAIL' END AS status,
  CONCAT('passenger_id nulls: ', CAST(SUM(CASE WHEN passenger_id IS NULL THEN 1 ELSE 0 END) AS STRING),
         ', route_id nulls: ', CAST(SUM(CASE WHEN route_id IS NULL THEN 1 ELSE 0 END) AS STRING),
         ', airline_id nulls: ', CAST(SUM(CASE WHEN airline_id IS NULL THEN 1 ELSE 0 END) AS STRING),
         ', nationkey nulls: ', CAST(SUM(CASE WHEN nationkey IS NULL THEN 1 ELSE 0 END) AS STRING)) AS details,
  'CRITICAL' AS severity
FROM sky_dev.gold.fact_booking;

SELECT * FROM dq_check_3;

In [0]:
-- CHECK 4: Numeric measures satisfy documented business rules
CREATE OR REPLACE TEMP VIEW dq_check_4 AS
SELECT
  'Check 4: Business Rules' AS check_name,
  CASE WHEN SUM(CASE WHEN ticket_price <= 0 THEN 1 ELSE 0 END) = 0
        AND SUM(CASE WHEN discount < 0 OR discount > 1 THEN 1 ELSE 0 END) = 0
        AND SUM(CASE WHEN seats < 1 THEN 1 ELSE 0 END) = 0
       THEN 'PASS' ELSE 'FAIL' END AS status,
  CONCAT('ticket_price<=0: ', CAST(SUM(CASE WHEN ticket_price <= 0 THEN 1 ELSE 0 END) AS STRING),
         ', discount outside [0,1]: ', CAST(SUM(CASE WHEN discount < 0 OR discount > 1 THEN 1 ELSE 0 END) AS STRING),
         ', seats<1: ', CAST(SUM(CASE WHEN seats < 1 THEN 1 ELSE 0 END) AS STRING)) AS details,
  'CRITICAL' AS severity
FROM sky_dev.gold.fact_booking;

SELECT * FROM dq_check_4;

In [0]:
-- CHECK 5: Gold revenue and row counts reconcile with Silver (tolerance ≤ 0.01%)
CREATE OR REPLACE TEMP VIEW dq_check_5 AS
WITH silver_stats AS (
  SELECT
    COUNT(*) AS s_rows,
    SUM(fb_ticketprice * (1 - fb_discount)) AS s_revenue
  FROM sky_dev.silver.flight_bookings fb
  JOIN sky_dev.silver.bookings b ON fb.fb_bookingid = b.bk_bookingid
  JOIN sky_dev.silver.passengers p ON b.bk_passengerid = p.pa_passengerid
),
gold_stats AS (
  SELECT
    COUNT(*) AS g_rows,
    SUM(ticket_price * (1 - discount)) AS g_revenue
  FROM sky_dev.gold.fact_booking
),
recon AS (
  SELECT
    s_rows, g_rows, s_revenue, g_revenue,
    CASE WHEN s_rows > 0 THEN ABS(g_rows - s_rows) / s_rows ELSE 0 END AS row_diff_pct,
    CASE WHEN s_revenue > 0 THEN ABS(g_revenue - s_revenue) / s_revenue ELSE 0 END AS rev_diff_pct
  FROM silver_stats s CROSS JOIN gold_stats g
)
SELECT
  'Check 5: Gold-Silver Reconciliation' AS check_name,
  CASE WHEN row_diff_pct <= 0.0001 AND rev_diff_pct <= 0.0001 THEN 'PASS' ELSE 'FAIL' END AS status,
  CONCAT('Rows — silver: ', CAST(s_rows AS STRING), ', gold: ', CAST(g_rows AS STRING),
         ', diff: ', CAST(ROUND(row_diff_pct * 100, 4) AS STRING), '%',
         ' | Revenue — silver: ', CAST(ROUND(s_revenue, 2) AS STRING),
         ', gold: ', CAST(ROUND(g_revenue, 2) AS STRING),
         ', diff: ', CAST(ROUND(rev_diff_pct * 100, 4) AS STRING), '%') AS details,
  'CRITICAL' AS severity
FROM recon;

SELECT * FROM dq_check_5;

In [0]:
-- CHECK 6: Rejected-record counts per table (informational)
CREATE OR REPLACE TEMP VIEW dq_check_6 AS
WITH reject_counts AS (
  SELECT 'airlines' AS tbl, COUNT(*) AS cnt FROM sky_dev.silver.quarantine_airline
  UNION ALL
  SELECT 'airline_routes', COUNT(*) FROM sky_dev.silver.quarantine_airline_routes
  UNION ALL
  SELECT 'bookings', COUNT(*) FROM sky_dev.silver.quarantine_bookings
  UNION ALL
  SELECT 'flight_bookings', COUNT(*) FROM sky_dev.silver.quarantine_flight_bookings
  UNION ALL
  SELECT 'nations', COUNT(*) FROM sky_dev.silver.quarantine_nations
  UNION ALL
  SELECT 'passengers', COUNT(*) FROM sky_dev.silver.quarantine_passengers
  UNION ALL
  SELECT 'regions', COUNT(*) FROM sky_dev.silver.quarantine_regions
  UNION ALL
  SELECT 'routes', COUNT(*) FROM sky_dev.silver.quarantine_routes
)
SELECT
  'Check 6: Rejected Records' AS check_name,
  'REPORTED' AS status,
  CONCAT_WS(', ', COLLECT_LIST(CONCAT(tbl, ': ', CAST(cnt AS STRING)))) AS details,
  'WARNING' AS severity
FROM reject_counts;

SELECT * FROM dq_check_6;

In [0]:
%python
# Aggregate all DQ results and raise pipeline-stopping error on critical failures
checks = []
for i in range(1, 7):
    rows = spark.sql(f"SELECT * FROM dq_check_{i}").collect()
    checks.extend(rows)

print("=" * 80)
print("DATA QUALITY CHECK RESULTS")
print("=" * 80)
for r in checks:
    icon = "PASS" if r.status == "PASS" else ("FAIL" if r.status == "FAIL" else "INFO")
    print(f"\n[{icon}] {r.check_name}")

critical_failures = [r for r in checks if r.severity == 'CRITICAL' and r.status == 'FAIL']
if critical_failures:
    failed_names = ", ".join(r.check_name for r in critical_failures)
    print("\n" + "=" * 80)
    print(f"PIPELINE STOPPED: {len(critical_failures)} critical check(s) failed")
    print(f"Failed: {failed_names}")
    print("=" * 80)
    raise Exception(f"Data quality pipeline stopped — {len(critical_failures)} critical check(s) failed: {failed_names}")
else:
    print("\n" + "=" * 80)
    print("ALL CRITICAL CHECKS PASSED — pipeline may proceed")
    print("=" * 80)